In [ ]:
import numpy as np


def autocorr_fft(x: np.ndarray) -> np.ndarray:
    """
    Fast autocorrelation using FFT.
    Returns acf[0..N-1] normalized so that acf[0] == 1 (if variance>0).
    """
    x = np.asarray(x, dtype=float)
    x = x - np.mean(x)
    n = x.size
    if n < 2:
        return np.ones(n)

    var = np.dot(x, x) / n
    if var == 0:
        return np.ones(n)

    # Zero-pad to 2n for circular convolution avoidance
    nfft = 1 << (2 * n - 1).bit_length()
    fx = np.fft.rfft(x, n=nfft)
    acf = np.fft.irfft(fx * np.conj(fx), n=nfft)[:n]
    # Unbiased normalization (divide by number of pairs at each lag)
    acf /= np.arange(n, 0, -1)
    # Normalize to 1 at lag 0
    acf /= acf[0]
    return acf


def tau_1_over_e(acf: np.ndarray, dt: float = 1.0) -> float:
    """
    tau_{1/e}: first lag where acf <= exp(-1).
    If it never crosses, returns np.nan.
    """
    acf = np.asarray(acf, dtype=float)
    thr = np.exp(-1.0)

    # Start from lag 1; lag 0 is 1 by definition
    idx = np.where(acf[1:] <= thr)[0]
    if idx.size == 0:
        return np.nan
    k = int(idx[0] + 1)
    return k * dt


def tau_integral(acf: np.ndarray, dt: float = 1.0, cutoff: str = "zero_crossing") -> float:
    """
    Integral time scale tau_int = ∫_0^∞ acf(tau) d tau (discrete sum).
    Because finite samples become noisy, we cut off the sum.

    cutoff options:
      - "zero_crossing": integrate until first non-positive lag (recommended default)
      - "1over_e": integrate until first lag where acf <= exp(-1)
      - "full": integrate all lags (not recommended; can be noisy / biased)
      - integer K: integrate lags 0..K inclusive (you can pass cutoff=K)
    """
    acf = np.asarray(acf, dtype=float)

    if isinstance(cutoff, (int, np.integer)):
        K = int(cutoff)
        K = max(0, min(K, acf.size - 1))
        return dt * np.sum(acf[: K + 1])

    if cutoff == "zero_crossing":
        # first lag >=1 where acf <= 0
        idx = np.where(acf[1:] <= 0.0)[0]
        K = int(idx[0] + 1) if idx.size > 0 else acf.size - 1
        return dt * np.sum(acf[: K + 1])

    if cutoff == "1over_e":
        thr = np.exp(-1.0)
        idx = np.where(acf[1:] <= thr)[0]
        K = int(idx[0] + 1) if idx.size > 0 else acf.size - 1
        return dt * np.sum(acf[: K + 1])

    if cutoff == "full":
        return dt * np.sum(acf)

    raise ValueError(f"Unknown cutoff='{cutoff}'")


def compute_time_scales(
    x: np.ndarray,
    dt: float = 1.0,
    tau_int_cutoff: str = "zero_crossing",
) -> dict:
    """
    Convenience wrapper:
    returns dict with acf, tau_1e, tau_int.
    """
    acf = autocorr_fft(x)
    return {
        "acf": acf,
        "tau_1e": tau_1_over_e(acf, dt=dt),
        "tau_int": tau_integral(acf, dt=dt, cutoff=tau_int_cutoff),
    }




In [2]:
ds = np.load("./data/l63_LATENT_GMM_dt1_T1000_w28_easy.npz")

In [ ]:
xys  = ds['trajs_train']


In [6]:
xys.shape

(40960, 1000, 3)

In [ ]:
# -------------------------
# Example usage
# -------------------------
# Suppose you have L63 trajectory arrays x_t, y_t, z_t sampled every dt=1 (your "Δt=1")
 # Choose a sample index (0, 1, ..., N-1)
# For example, if you have 100 samples of 200 time steps each:
# xys[sample, 100:200, 0] gives you the x_t, y_t, z_t for that sample
# Here we assume xys is a 3D array with shape (N_samples, N_timesteps, 3)
# where the last dimension corresponds to (x, y, z) coordinates.
# You can loop over samples and compute the time scales for each variable.
# For example, if you want to compute the time scales for the first 100 samples:
tau_1e_x = []   
tau_1e_y = []
tau_1e_z = []
tau_int_x = []   
tau_int_y = []
tau_int_z = []

for sample in range(1000):   
    ############# IGNORE THIS LINE #############
    x_t = xys[sample, 100:200,0]
    y_t = xys[sample, 100:200,1]
    z_t = xys[sample, 100:200,2]
    dt = 1

    x_t = xys[sample, 100:200,0]
    y_t = xys[sample, 100:200,1]
    z_t = xys[sample, 100:200,2]
    dt = 1

    scales_x = compute_time_scales(x_t, dt=dt)
    scales_y = compute_time_scales(y_t, dt=dt)
    scales_z = compute_time_scales(z_t, dt=dt)
    tau_1e_x.append(scales_x["tau_1e"])
    tau_1e_y.append(scales_y["tau_1e"])
    tau_1e_z.append(scales_z["tau_1e"])
    tau_int_x.append(scales_x["tau_int"])
    tau_int_y.append(scales_y["tau_int"])
    tau_int_z.append(scales_z["tau_int"])

    # print("x:", scales_x["tau_1e"], scales_x["tau_int"])
    # print("y:", scales_y["tau_1e"], scales_y["tau_int"])
    # print("z:", scales_z["tau_1e"], scales_z["tau_int"])

    # # If you want one scalar "system turnover time", a conservative choice is the max across variables:
    # tau_1e_system  = np.nanmax([scales_x["tau_1e"],  scales_y["tau_1e"],  scales_z["tau_1e"]])
    # tau_int_system = np.nanmax([scales_x["tau_int"], scales_y["tau_int"], scales_z["tau_int"]])
    # print("system:", tau_1e_system, tau_int_system)


In [18]:
print ("x:", np.nanmean(tau_1e_x), np.nanmean(tau_int_x))
print ("y:", np.nanmean(tau_1e_y), np.nanmean(tau_int_y))
print ("z:", np.nanmean(tau_1e_z), np.nanmean(tau_int_z))


x: 12.093 10.23552430641024
y: 9.867 8.504833800830216
z: 8.382 7.066818905114636
